# PySpark Mylib

- It has 2 techniques 
        1. RDD API Technique
        2. DataFRame API (Recent one)

In [40]:
import pyspark
import pandas as pd

In [53]:
df = pd.read_csv("test3.csv")
df

,name,age,Experience,Salary
0,Prabha,32,20,10000
1,Yogesh,33,10,20000
2,Atharva,2,5,30000
3,Pallu,4,10,40000
4,Pavani,10,5,50000
5,Pranamya,6,3,60000


In [54]:
type(df)

pandas.core.frame.DataFrame

## Start SPARK Session with session name "Practice"

In [55]:
from pyspark.sql  import SparkSession

In [56]:
spark = SparkSession.builder.appName('Practice1').getOrCreate()

In [57]:
spark

In [58]:
## Read Datset with respect to spark

In [60]:
## By Default csv file all column read as string, so use inferSchema while reading
training = spark.read.option('header','true').csv('test3.csv', inferSchema='true')
training.show()

+--------+---+----------+------+
|    name|age|Experience|Salary|
+--------+---+----------+------+
|  Prabha| 32|        20| 10000|
|  Yogesh| 33|        10| 20000|
|Atharva |  2|         5| 30000|
|   Pallu|  4|        10| 40000|
|  Pavani| 10|         5| 50000|
|Pranamya|  6|         3| 60000|
+--------+---+----------+------+



In [61]:
training.printSchema()

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)



In [62]:
training.columns

['name', 'age', 'Experience', 'Salary']

> In Python while training we do test train split etc and before that we do preprocessing , But in PySpark we will do Vectorassembly

> This time we just assemple the independent feature as separte

> ['age', 'Experience'] --> New Feature --> Independent Feature 

In [63]:
from pyspark.ml.feature import VectorAssembler 
featureassembler = VectorAssembler(inputCols = ['age', 'Experience'] ,outputCol = 'Independent Features')

In [64]:
output = featureassembler.transform(training)

In [65]:
output.show()
# Here combined col created as vector(Grouped independent feature)

+--------+---+----------+------+--------------------+
|    name|age|Experience|Salary|Independent Features|
+--------+---+----------+------+--------------------+
|  Prabha| 32|        20| 10000|         [32.0,20.0]|
|  Yogesh| 33|        10| 20000|         [33.0,10.0]|
|Atharva |  2|         5| 30000|           [2.0,5.0]|
|   Pallu|  4|        10| 40000|          [4.0,10.0]|
|  Pavani| 10|         5| 50000|          [10.0,5.0]|
|Pranamya|  6|         3| 60000|           [6.0,3.0]|
+--------+---+----------+------+--------------------+



> Now new col "Independent Features" is an input feature and "Salary" is our output feature

In [68]:
output.columns

['name', 'age', 'Experience', 'Salary', 'Independent Features']

In [69]:
finalized_data = output.select('Independent Features','Salary')

In [70]:
finalized_data.show()

+--------------------+------+
|Independent Features|Salary|
+--------------------+------+
|         [32.0,20.0]| 10000|
|         [33.0,10.0]| 20000|
|           [2.0,5.0]| 30000|
|          [4.0,10.0]| 40000|
|          [10.0,5.0]| 50000|
|           [6.0,3.0]| 60000|
+--------------------+------+



## Do Traintest Split and Apply Model 

In [83]:
from pyspark.ml.regression import LinearRegression

#traintest split
train_data, test_data=finalized_data.randomSplit([0.75,0.25])

#In Pyspark, during model initialization time only input(featuresCol) and output(labelCol) col are defined. So no need to split like train_X,train_Y like pandas
regressor = LinearRegression(featuresCol='Independent Features', labelCol = 'Salary')
regressor = regressor.fit(train_data)

Exception ignored in: <function JavaWrapper.__del__ at 0x00000283EC627158>
Traceback (most recent call last):
  File "C:\Users\prabh\Anaconda3\lib\site-packages\pyspark\ml\wrapper.py", line 39, in __del__
    if SparkContext._active_spark_context and self._java_obj is not None:
AttributeError: 'VectorAssembler' object has no attribute '_java_obj'
Exception ignored in: <function JavaWrapper.__del__ at 0x00000283EC627158>
Traceback (most recent call last):
  File "C:\Users\prabh\Anaconda3\lib\site-packages\pyspark\ml\wrapper.py", line 39, in __del__
    if SparkContext._active_spark_context and self._java_obj is not None:
AttributeError: 'LinearRegression' object has no attribute '_java_obj'


In [84]:
test_data.show()

+--------------------+------+
|Independent Features|Salary|
+--------------------+------+
|         [32.0,20.0]| 10000|
|         [33.0,10.0]| 20000|
+--------------------+------+



In [85]:
## coefficients
regressor.coefficients

DenseVector([2245.3704, -1342.5926])

In [86]:
## intercept
regressor.intercept

40370.370370370416

In [88]:
## Prediction
pred_results = regressor.evaluate(test_data)

#If test set is empty we will get error message like "IllegalArgumentException: requirement failed: Nothing has been added to this summarizer."

In [91]:
#predictions is a variable in o/p, this will have op
pred_results.predictions.show()
# In Pyspark, it combines predicted o/p with orignal data and provide, but in Normal pandas it wont like this

+--------------------+------+------------------+
|Independent Features|Salary|        prediction|
+--------------------+------+------------------+
|         [32.0,20.0]| 10000| 85370.37037037022|
|         [33.0,10.0]| 20000|101041.66666666654|
+--------------------+------+------------------+



In [96]:
#MAE
pred_results.meanAbsoluteError

78206.01851851838

In [97]:
#MAE, MSE, r2
pred_results.meanAbsoluteError,pred_results.meanSquaredError ,pred_results.r2

(78206.01851851838, 6124222232.938936)